# 02 - VECM and Johansen Cointegration Test

This notebook covers **cointegration** analysis and the **Vector Error Correction Model (VECM)**
using `chronobox`.

## Topics covered

1. Cointegration: long-run equilibrium relationships
2. Johansen test (trace and max-eigenvalue statistics)
3. VECM estimation and interpretation
4. Alpha (speed of adjustment) and Beta (cointegrating vectors)
5. The long-run matrix $\Pi = \alpha \beta'$
6. Exercises

---

### From VAR to VECM

If $K$ variables are **cointegrated** (share common stochastic trends), the VAR in levels
can be reparameterized as a VECM:

$$\Delta Y_t = \Pi Y_{t-1} + \Gamma_1 \Delta Y_{t-1} + \cdots + \Gamma_{p-1} \Delta Y_{t-p+1} + c + u_t$$

where:
- $\Pi = \alpha \beta'$ is the **long-run matrix** of rank $r < K$
- $\beta$ contains the $r$ **cointegrating vectors** (long-run equilibrium relations)
- $\alpha$ contains the **loading coefficients** (speed of adjustment to equilibrium)
- $\Gamma_i$ capture short-run dynamics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox import VECM

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_multivariate_series

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading the data

We use the Canadian macroeconomic dataset. These variables are likely I(1) and may
share cointegrating relationships.

In [ ]:
data_path = os.path.join("..", "data", "canada_macro.csv")
df = pd.read_csv(data_path)
var_names = ["e", "prod", "rw", "U"]
endog = df[var_names].values

fig = plot_multivariate_series(df, date_col="date", title="Canadian Macro Variables")
plt.show()

print(f"Shape: {endog.shape} ({endog.shape[0]} obs, {endog.shape[1]} variables)")

## 2. Johansen cointegration test

The **Johansen test** determines the number of cointegrating relationships $r$ among
$K$ variables. It provides two test statistics:

- **Trace statistic**: $\lambda_{\text{trace}}(r) = -T \sum_{i=r+1}^{K} \ln(1 - \hat{\lambda}_i)$
  - Tests $H_0: \text{rank}(\Pi) \leq r$ vs $H_1: \text{rank}(\Pi) > r$

- **Max-eigenvalue statistic**: $\lambda_{\max}(r) = -T \ln(1 - \hat{\lambda}_{r+1})$
  - Tests $H_0: \text{rank}(\Pi) = r$ vs $H_1: \text{rank}(\Pi) = r + 1$

We proceed sequentially: start with $r=0$ and increase until we fail to reject.

In [ ]:
# Run Johansen test with 2 lags in levels (1 lagged difference in VECM)
vecm = VECM(lags=2, deterministic="ci")
johansen = vecm.johansen_test(endog)

print(johansen.summary())

In [ ]:
# Detailed trace test results
K = endog.shape[1]
print("Trace Test:")
print(f"{'r':>4s}  {'Trace Stat':>12s}  {'90%':>10s}  {'95%':>10s}  {'99%':>10s}  {'Decision':>10s}")
print("-" * 70)
for r in range(K):
    stat = johansen.trace_stat[r]
    c90, c95, c99 = johansen.trace_crit[r]
    reject = "Reject" if stat > c95 else "Fail"
    print(f"{r:>4d}  {stat:>12.4f}  {c90:>10.4f}  {c95:>10.4f}  {c99:>10.4f}  {reject:>10s}")

print(f"\nTrace test suggests rank = {johansen.rank_trace}")

In [ ]:
# Max-eigenvalue test
print("Max-Eigenvalue Test:")
print(f"{'r':>4s}  {'Max-Eig Stat':>14s}  {'90%':>10s}  {'95%':>10s}  {'99%':>10s}  {'Decision':>10s}")
print("-" * 72)
for r in range(K):
    stat = johansen.max_eig_stat[r]
    c90, c95, c99 = johansen.max_eig_crit[r]
    reject = "Reject" if stat > c95 else "Fail"
    print(f"{r:>4d}  {stat:>14.4f}  {c90:>10.4f}  {c95:>10.4f}  {c99:>10.4f}  {reject:>10s}")

print(f"\nMax-eigenvalue test suggests rank = {johansen.rank_maxeig}")

In [ ]:
# Plot eigenvalues
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, K + 1), johansen.eigenvalues, color="steelblue", alpha=0.8)
ax.set_xlabel("Eigenvalue index")
ax.set_ylabel("Eigenvalue")
ax.set_title("Johansen Eigenvalues")
ax.set_xticks(range(1, K + 1))
ax.grid(True, alpha=0.3, axis="y")

for i, ev in enumerate(johansen.eigenvalues):
    ax.text(i + 1, ev + 0.005, f"{ev:.4f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "johansen_eigenvalues.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:** The number of cointegrating relations tells us how many
independent long-run equilibrium relationships exist among the variables. With $K=4$
variables and $r$ cointegrating vectors, there are $K - r$ common stochastic trends
(permanent shocks). If $r=1$, three common trends drive the system and one equilibrium
relationship ties the variables together in the long run.

## 3. Fitting the VECM

We now estimate the VECM using the cointegration rank determined by the Johansen test.

In [ ]:
# Fit VECM with the rank from the trace test
r = johansen.rank_trace
print(f"Fitting VECM with cointegration rank r = {r}")

vecm_model = VECM(lags=2, coint_rank=r, deterministic="ci")
vecm_results = vecm_model.fit(endog, names=var_names)

print(vecm_results.summary())

## 4. Interpreting Alpha and Beta

### Beta (cointegrating vectors)

$\beta' Y_t$ defines the long-run equilibrium. When $\beta' Y_t \neq 0$, the system
is out of equilibrium, and the error correction mechanism pulls it back.

### Alpha (loading/adjustment coefficients)

$\alpha$ governs how fast each variable adjusts to the equilibrium deviation.
A negative $\alpha_i$ means variable $i$ decreases when $\beta' Y_t > 0$ (error
correcting behavior). A positive $\alpha_i$ means the variable moves away (destabilizing).

In [ ]:
# Display alpha (adjustment speeds)
alpha_df = pd.DataFrame(
    vecm_results.alpha,
    index=var_names,
    columns=[f"EC{i+1}" for i in range(vecm_results.coint_rank)]
)
print("Alpha (speed of adjustment):")
print(alpha_df.round(4))

print("\n--- Interpretation ---")
for i, name in enumerate(var_names):
    for j in range(vecm_results.coint_rank):
        a = vecm_results.alpha[i, j]
        direction = "error-correcting" if a < 0 else "destabilizing"
        print(f"  {name} responds to EC{j+1}: alpha = {a:.4f} ({direction})")

In [ ]:
# Display beta (cointegrating vectors)
beta_df = pd.DataFrame(
    vecm_results.beta,
    columns=[f"EC{i+1}" for i in range(vecm_results.coint_rank)]
)
# Beta may have extra rows for deterministic terms
beta_labels = var_names.copy()
if vecm_results.beta.shape[0] > len(var_names):
    for d in range(vecm_results.beta.shape[0] - len(var_names)):
        beta_labels.append(f"det_{d}")
beta_df.index = beta_labels

print("Beta (cointegrating vectors):")
print(beta_df.round(4))

print("\nLong-run equilibrium relation(s):")
for j in range(vecm_results.coint_rank):
    terms = []
    for i, name in enumerate(var_names):
        coef = vecm_results.beta[i, j]
        if abs(coef) > 1e-10:
            terms.append(f"{coef:+.4f}*{name}")
    print(f"  EC{j+1}: {' '.join(terms)} = 0")

In [ ]:
# Plot the error correction terms over time
# Compute beta'*Y_t for the cointegrating relations
beta_vars = vecm_results.beta[:len(var_names), :]  # only variable part
ec_terms = endog @ beta_vars  # shape (T, r)

fig, axes = plt.subplots(vecm_results.coint_rank, 1, figsize=(12, 3 * vecm_results.coint_rank),
                         squeeze=False)
fig.suptitle("Error Correction Terms Over Time", fontsize=14)

dates = pd.to_datetime(df["date"])
for j in range(vecm_results.coint_rank):
    ax = axes[j, 0]
    ax.plot(dates, ec_terms[:, j], color="steelblue", linewidth=1.0)
    ax.axhline(0, color="red", linestyle="--", linewidth=0.8)
    ax.set_ylabel(f"EC{j+1}")
    ax.set_title(f"Cointegrating Relation {j+1}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "ec_terms.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:** The error correction terms should fluctuate around zero.
Persistent deviations indicate that the system is temporarily out of long-run equilibrium.
The speed of adjustment coefficients ($\alpha$) determine how quickly each variable
corrects back. Variables with small $|\alpha|$ respond slowly; those with large $|\alpha|$
adjust quickly to restore equilibrium.

In [ ]:
# The long-run impact matrix Pi = alpha * beta'
pi = vecm_results.pi
print("Long-run matrix Pi = alpha * beta':")
print(pd.DataFrame(pi, index=var_names, columns=var_names).round(4))

print(f"\nRank of Pi: {vecm_results.coint_rank}")
print(f"Number of common stochastic trends: {len(var_names) - vecm_results.coint_rank}")

In [ ]:
# Display short-run dynamics (Gamma matrices)
for i, gamma in enumerate(vecm_results.gamma):
    print(f"\nGamma_{i+1} (short-run dynamics, lagged difference {i+1}):")
    print(pd.DataFrame(gamma, index=var_names, columns=var_names).round(4))

## Exercicio

### Exercicio 1: Cointegracao nos dados US Macro

Use o dataset `us_macro_quarterly.csv` para:

1. Realizar o teste de Johansen com `lags=4` e `deterministic="ci"`
2. Comparar os resultados trace vs max-eigenvalue
3. Estimar o VECM com o rank encontrado
4. Interpretar os vetores de cointegracao ($\beta$)
5. Quais variaveis se ajustam mais rapido ao equilibrio?

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

us_df = pd.read_csv(os.path.join("..", "data", "us_macro_quarterly.csv"))
us_names = ["gdp", "inflation", "fed_funds", "unemployment"]
us_endog = us_df[us_names].values

# Johansen test
us_vecm = VECM(lags=4, deterministic="ci")
us_johansen = us_vecm.johansen_test(us_endog)
print(us_johansen.summary())

# Fit VECM
r_us = us_johansen.rank_trace
us_vecm_fit = VECM(lags=4, coint_rank=r_us, deterministic="ci")
us_vecm_results = us_vecm_fit.fit(us_endog, names=us_names)
print(us_vecm_results.summary())

# Alpha interpretation
alpha_us = pd.DataFrame(
    us_vecm_results.alpha,
    index=us_names,
    columns=[f"EC{i+1}" for i in range(r_us)]
)
print("\nAlpha (US data):")
print(alpha_us.round(4))

### Exercicio 2: Efeito da especificacao deterministica

Compare os resultados do teste de Johansen nos dados canadenses com diferentes
especificacoes deterministicas:
- `deterministic="nc"` (sem constante)
- `deterministic="ci"` (constante dentro do ECM)
- `deterministic="co"` (constante fora do ECM)

Como a escolha afeta o rank estimado?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

for det in ["nc", "ci", "co"]:
    v = VECM(lags=2, deterministic=det)
    j = v.johansen_test(endog)
    print(f"Deterministic='{det}': Trace rank={j.rank_trace}, Max-eig rank={j.rank_maxeig}")

---

## Resumo

Neste notebook aprendemos:

1. **Cointegracao** captura relacoes de equilibrio de longo prazo entre series I(1)
2. O **teste de Johansen** (trace e max-eigenvalue) determina o numero de relacoes de cointegracao
3. O **VECM** decompoe a dinamica em componentes de longo prazo ($\Pi = \alpha\beta'$) e curto prazo ($\Gamma_i$)
4. **Beta** define as relacoes de equilibrio; **alpha** mede a velocidade de ajuste
5. A escolha da especificacao deterministica afeta os resultados do teste

No proximo notebook, analisaremos **funcoes impulso-resposta (IRF)** e **decomposicao da variancia (FEVD)**.